# 00B — Samples, Tokens, Batches, and Local Training Data

## What these terms mean

A **sample** is a dataset record, such as a document or conversation. A **token** is a vocabulary unit, not necessarily a whole word. A **tokenizer** converts text to integer token IDs. A **chat template** inserts the role and turn markers expected by a checkpoint.

A **batch** groups samples processed together. An **epoch** is one pass over the training data. A **microbatch step** processes one batch; an **optimizer step** updates parameters. **Gradient accumulation** combines gradients from several microbatches before an optimizer step. Check which “step” a log reports.

A **label** is a prediction target. A **collator** combines samples into tensors and adds **padding** to make lengths compatible. **End-of-sequence (EOS)** marks a sequence ending; EOS and padding may share an ID, so ID alone cannot distinguish them.

**What problem does this solve?** Explicit conversion gives local columns a precise training meaning. A valid file is not necessarily a correctly supervised dataset.


## NovaBot: raw file to canonical record

**Hand-constructed fictional example:**

~~~csv
group_id,question,answer
manual-1,How many modes does NovaBot have?,Three: idle mapping and navigation.
~~~

Convert it to a conversation:

~~~json
{"group_id":"manual-1","messages":[
  {"role":"user","content":"How many modes does NovaBot have?"},
  {"role":"assistant","content":"Three: idle, mapping, navigation."}
]}
~~~

**CSV** means comma-separated values. **JSON** means JavaScript Object Notation, a structured text format. **JSON Lines (JSONL)** stores one JSON value per line; flatten the displayed record to one line for JSONL. group_id preserves shared provenance, not necessarily model input.

The path is raw fields → canonical record → rendered chat text → token IDs → padded batch. An image path is a file reference, not a learned image representation.


## Attention masks versus loss masks

An **attention mask** marks real positions (1) versus padding (0). The model's causal rules additionally prevent reading future tokens. A **loss mask** selects scored targets. This course uses label -100 to ignore a target; -100 is not a vocabulary token.

**Supervised Fine-Tuning (SFT)** learns demonstrated answers. Typically the model reads the prompt but scores only assistant targets:

| Illustrative position | Attention | Stored label |
|---|---:|---|
| User prompt token | 1 | -100 |
| Answer token “Three” | 1 | Its token ID |
| Real assistant EOS | 1 | EOS token ID |
| Padding | 0 | -100 |

The output at position $t$ predicts the label at $t+1$, where $t$ indexes token positions. Labels are stored unshifted; the model performs the causal shift. Do not shift twice.

**Hand calculation:** 10 examples, microbatch size 2, and accumulation of 2 microbatches give 5 microbatches and 3 optimizer updates per epoch. The last update has only one microbatch. Later loops weight losses by valid target count, including that final short window.


## Data leakage and connection to the cells

**Training data** updates parameters. **Validation data** helps choose settings. **Test data** supports the final assessment. **Data leakage** occurs when training or model selection sees information that an assessment is supposed to hold out.

Copies or chunks of NovaBot's same manual can leak across splits even with different filenames. Deduplicate, retain source groups, split groups, and only then chunk long documents.

raw_rows reads local files; records assigns canonical meanings; split_records() deduplicates and groups. encode_conversation() creates assistant labels; collate_text() pads tensors; token_table() displays them. The runnable color/arithmetic fixtures remain unchanged; the NovaBot example explains their conversion pattern. Image processing is expanded in lesson 07.


## Common confusions and quick check

“Unlabeled text” can still supply next-token targets automatically. Ignoring prompt loss does not mean deleting the prompt.

1. Must a token with label -100 also have attention mask 0?
2. Should two paraphrased questions from one source manual be split independently?

<details>
<summary>Answers</summary>

1. No. A prompt can remain visible with attention 1 while its label is ignored.
2. No. Keep the source group together; different wording does not make the evidence independent.

</details>


## Before running the experiment

**Learning goals:** convert raw TXT, CSV and JSONL; preserve conversation/document provenance; inspect chat rendering, labels and image references.

**Prerequisites:** basic Python. The concepts needed for this lesson are introduced above. Run every cell in order in a fresh kernel. The default `tiny_cpu` mode has random weights and a synthetic vocabulary: output quality is not evidence of Qwen's capabilities. Use `local_pretrained` for an already downloaded checkpoint.

[Course index](README.md) · [Execution and data flow](../docs/EXECUTION_AND_DATA_FLOW.md)

**Experiment contract:** inspect inputs before training, keep held-out records separate, check gradients/parameter changes, then save and reload. Each notebook is independent.

[Terminology reference](../docs/GLOSSARY.md) · [Compare training methods](../docs/TRAINING_METHODS.md)


## Choose the experiment

The model directory must contain its own weights, tokenizer, processor and chat template. The environment variables below are optional; edit the parameter cell directly in Jupyter. Files created by this lesson stay under its output directory.

For local weights, set `FTLAB_DEVICE` to `cpu`, `mps`, `cuda`, or `auto` before launching. CPU/MPS default to ordinary LoRA; CUDA retains its QLoRA profile. Restart the kernel when changing devices after a Trainer has initialized. Selecting a device does not guarantee the full experiment fits its memory.


In [1]:
LESSON = "00b"
# Parameters: change these before running the notebook from top to bottom.
import csv
import json
import os
import random
from pathlib import Path

import numpy as np
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor

from finetunelab.devices import (
    activate_runtime,
    resolve_runtime,
)
from finetunelab.education import (
    inspect_local_checkpoint,
    make_tiny_checkpoint,
    project_root,
    token_table,
)
from finetunelab.tuning import parameter_report

MODE = os.environ.get("FTLAB_NOTEBOOK_MODE", "tiny_cpu")
LOCAL_MODEL_PATH = Path(os.environ.get("FTLAB_LOCAL_MODEL", "models/Qwen3.5-2B"))
LOCAL_TEACHER_PATH = Path(os.environ.get("FTLAB_LOCAL_TEACHER", "models/Qwen3.5-4B"))
ROOT = project_root()
DATA_ROOT = Path(os.environ.get("FTLAB_LESSON_DATA", str(ROOT / "examples/education")))
OUTPUT_ROOT = Path(os.environ.get("FTLAB_NOTEBOOK_OUTPUT", str(ROOT / "outputs/notebooks")))
OUTPUT = OUTPUT_ROOT / LESSON
OUTPUT.mkdir(parents=True, exist_ok=True)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(2)
assert MODE in {"tiny_cpu", "local_pretrained"}
# tiny_cpu remains an offline CPU fixture; real checkpoints use the selected backend.
REQUESTED_DEVICE = "cpu" if MODE == "tiny_cpu" else os.environ.get("FTLAB_DEVICE", "auto")
RUNTIME = resolve_runtime(device=REQUESTED_DEVICE, dtype=os.environ.get("FTLAB_DTYPE"))
activate_runtime(RUNTIME)
DEVICE = torch.device(RUNTIME.device)
DTYPE = RUNTIME.torch_dtype
ATTENTION = "eager" if MODE == "tiny_cpu" else RUNTIME.attention
print(RUNTIME.report())
MODEL_PATH = (
    make_tiny_checkpoint(OUTPUT / "initial", seed=SEED)
    if MODE == "tiny_cpu"
    else LOCAL_MODEL_PATH.expanduser().resolve()
)
checkpoint_info = inspect_local_checkpoint(MODEL_PATH)
print({"mode": MODE, "device": str(DEVICE), "checkpoint": str(MODEL_PATH)})
print(checkpoint_info["files"])
if DEVICE.type == "mps":
    torch.mps.manual_seed(SEED)

/Users/haroldye/Desktop/Code/FineTuneLab/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'requested': 'mps', 'device': 'mps', 'dtype': 'bfloat16', 'attention': 'eager', 'bf16': False, 'fp16': False, 'tf32': False}
{'mode': 'local_pretrained', 'device': 'mps', 'checkpoint': '/Users/haroldye/Desktop/Code/FineTuneLab/notebooks/models/Qwen3.5-2B'}
['.gitattributes', 'LICENSE', 'README.md', 'chat_template.jinja', 'config.json', 'merges.txt', 'model.safetensors-00001-of-00001.safetensors', 'model.safetensors.index.json', 'preprocessor_config.json', 'tokenizer.json', 'tokenizer_config.json', 'video_preprocessor_config.json', 'vocab.json']


## Load the local checkpoint

`from_pretrained()` constructs the official PyTorch modules and fills their tensors from the checkpoint. `local_files_only=True` prevents a missing local file from becoming a network download. BF16 and NF4 serve different roles: computation precision versus storage of the frozen base.


In [2]:
# A checkpoint includes both weights and the preprocessing contract.
processor = AutoProcessor.from_pretrained(MODEL_PATH, local_files_only=True)
tokenizer = processor.tokenizer
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
load_kwargs = {
    "local_files_only": True,
    "dtype": DTYPE,
    "device_map": {"": str(DEVICE)},
    "attn_implementation": ATTENTION,
}
# Quantization is a separate choice. CPU/MPS use unquantized LoRA.
_default_qlora = (
    MODE == "local_pretrained" and DEVICE.type == "cuda" and LESSON not in {"00a", "00b", "04"}
)
USE_QLORA = os.environ.get("FTLAB_USE_QLORA", str(_default_qlora)).lower() in {"1", "true", "yes"}
if USE_QLORA and DEVICE.type != "cuda":
    raise ValueError("This CPU/MPS profile supports ordinary LoRA; set FTLAB_USE_QLORA=false.")
if USE_QLORA:
    from transformers import BitsAndBytesConfig

    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )
    load_kwargs["device_map"] = {"": torch.cuda.current_device()}
model = AutoModelForImageTextToText.from_pretrained(MODEL_PATH, **load_kwargs)
if not USE_QLORA:
    model.to(DEVICE)
model.config.use_cache = False
print(type(model).__name__, parameter_report(model))

Loading weights: 100%|██████████████████████| 617/617 [00:02<00:00, 276.97it/s]


Qwen3_5ForConditionalGeneration {'total': 2213241664, 'trainable': 2213241664, 'frozen': 0, 'trainable_percent': 100.0}


## From local Q&A to train/validation/test records

A dataset row is not yet a tensor. Preserve the original group identity so examples from one conversation stay together. These tiny held-out splits demonstrate plumbing; use representative, larger splits in real experiments.


In [3]:
# Convert local Q&A rows to canonical conversations; preserve provenance.
QA_FILE = Path(os.environ.get("FTLAB_QA_FILE", str(DATA_ROOT / "qa.csv")))
if QA_FILE.suffix.lower() == ".csv":
    with QA_FILE.open(encoding="utf-8", newline="") as handle:
        raw_rows = list(csv.DictReader(handle))
elif QA_FILE.suffix.lower() == ".jsonl":
    raw_rows = [
        json.loads(line)
        for line in QA_FILE.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
else:
    raise ValueError("This converter accepts CSV or JSONL Q&A files.")
for row in raw_rows:
    if (
        not row.get("group_id")
        or not row.get("question", "").strip()
        or not row.get("answer", "").strip()
    ):
        raise ValueError("Every Q&A needs a group_id, nonempty question, and nonempty answer.")
    row.setdefault("rejected", "")
records = [
    {
        "group_id": row["group_id"],
        "messages": [
            {"role": "user", "content": row["question"]},
            {"role": "assistant", "content": row["answer"]},
        ],
        "prompt": row["question"],
        "chosen": row["answer"],
        "rejected": row["rejected"],
    }
    for row in raw_rows
]


def split_records(rows, seed=SEED):
    # Deduplicate before splitting. Never split one document/conversation group.
    unique = {}
    for row in rows:
        key = json.dumps(row["messages"], sort_keys=True, ensure_ascii=False)
        unique.setdefault(key, row)
    groups = sorted({row["group_id"] for row in unique.values()})
    if len(groups) < 3:
        raise ValueError("Provide at least three independent document/conversation groups.")
    random.Random(seed).shuffle(groups)
    validation_groups, test_groups = set(groups[:1]), set(groups[1:2])
    splits = {"train": [], "validation": [], "test": []}
    for row in unique.values():
        split = (
            "validation"
            if row["group_id"] in validation_groups
            else "test"
            if row["group_id"] in test_groups
            else "train"
        )
        splits[split].append(row)
    return splits


splits = split_records(records)
for split, rows in splits.items():
    with (OUTPUT / f"{split}.jsonl").open("w", encoding="utf-8") as handle:
        for row in rows:
            canonical = {"group_id": row["group_id"], "messages": row["messages"]}
            handle.write(json.dumps(canonical, ensure_ascii=False) + "\n")
print({split: len(rows) for split, rows in splits.items()})
print("Raw:", raw_rows[0])
print("Canonical:", records[0])

{'train': 10, 'validation': 1, 'test': 1}
Raw: {'group_id': 'sky', 'question': 'What is the color of sky ?', 'answer': 'blue', 'rejected': 'green'}
Canonical: {'group_id': 'sky', 'messages': [{'role': 'user', 'content': 'What is the color of sky ?'}, {'role': 'assistant', 'content': 'blue'}], 'prompt': 'What is the color of sky ?', 'chosen': 'blue', 'rejected': 'green'}


In [3]:
DATA_ROOT

PosixPath('/Users/haroldye/Desktop/Code/FineTuneLab/examples/education')

## Chat rendering, tokenization and loss masking

The chat template supplies role delimiters. Attention masks describe real positions versus padding; labels choose prediction targets. A user token can be visible to attention while its label is `-100`. Assistant EOS should remain supervised even if its ID equals the padding ID.

Qwen3.5 renders earlier assistant turns differently from the final answer (including its thinking markers). The cell below adds `{% generation %}` annotations to the recognized checkpoint template in memory, preserving its rendered text. These annotations identify assistant targets directly in the full conversation, including each assistant EOS, without comparing separately rendered prefixes.


In [4]:
import re


def assistant_training_template(template):
    # Qwen3.5 omits reasoning from earlier turns, so re-rendered prefixes can change.
    # Annotate its assistant branch without changing the rendered checkpoint text.
    if re.search(r"{%-?\s*generation\s*-?%}", template):
        return template
    replacements = {
        '{%- if loop.index0 > ns.last_query_index %}': (
            "{{- '<|im_start|>' + message.role + '\\n' }}"
            "{%- generation %}{%- if loop.index0 > ns.last_query_index %}"
        ),
        "'<|im_start|>' + message.role + '\\n<think>\\n'": "'<think>\\n'",
        "{{- '<|im_start|>' + message.role + '\\n' + content }}": "{{- content }}",
        '{%- elif message.role == "tool" %}': (
            '{%- endgeneration %}{%- elif message.role == "tool" %}'
        ),
    }
    # Only adapt the recognized Qwen layout; other templates retain alignment checks.
    if not all(template.count(old) == 1 for old in replacements):
        return template
    for old, new in replacements.items():
        template = template.replace(old, new)
    return template


training_template = assistant_training_template(tokenizer.get_chat_template())


def render(messages, generation=False):
    return tokenizer.apply_chat_template(
        messages,
        chat_template=training_template,
        tokenize=False,
        add_generation_prompt=generation,
        enable_thinking=False,
    )


def encode_conversation(messages):
    # Generation spans select assistant output, including EOS in our templates.
    template = training_template
    if re.search(r"{%-?\s*generation\s*-?%}", template):
        encoded = tokenizer.apply_chat_template(
            messages,
            chat_template=template,
            tokenize=True,
            add_generation_prompt=False,
            return_dict=True,
            return_assistant_tokens_mask=True,
            enable_thinking=False,
        )
        ids = encoded["input_ids"]
        supervised = encoded["assistant_masks"]
    else:
        # For templates without generation annotations, verify prefix alignment.
        # Do not guess a token count by separately tokenizing the answer.
        ids = tokenizer(render(messages), add_special_tokens=False)["input_ids"]
        supervised = [0] * len(ids)
        for index, message in enumerate(messages):
            if message["role"] != "assistant":
                continue
            prefix = tokenizer(render(messages[:index], generation=True), add_special_tokens=False)[
                "input_ids"
            ]
            completed = tokenizer(render(messages[: index + 1]), add_special_tokens=False)[
                "input_ids"
            ]
            if ids[: len(prefix)] != prefix or ids[: len(completed)] != completed:
                raise ValueError(
                    "Template is not prefix-stable; use a training template with generation tags."
                )
            supervised[len(prefix) : len(completed)] = [1] * (len(completed) - len(prefix))
    if not any(supervised[1:]):
        raise ValueError("No assistant target tokens remain.")
    return {
        "input_ids": ids,
        "labels": [t if keep else -100 for t, keep in zip(ids, supervised, strict=False)],
    }


def collate_text(rows):
    items = [encode_conversation(row["messages"]) for row in rows]
    encoded = tokenizer.pad(
        [{"input_ids": item["input_ids"]} for item in items],
        padding=True,
        return_tensors="pt",
    )
    # Padding labels are independent of the pad token ID (pad may equal EOS).
    labels = torch.full_like(encoded["input_ids"], -100)
    for index, item in enumerate(items):
        labels[index, : len(item["labels"])] = torch.tensor(item["labels"])
    encoded["labels"] = labels
    return dict(encoded)


batch = collate_text(splits["train"][:2])
print(render(splits["train"][0]["messages"]))
print({name: tuple(value.shape) for name, value in batch.items()})
display(token_table(tokenizer, batch))

<|im_start|>user
What is the color of grass ?<|im_end|>
<|im_start|>assistant
<think>

</think>

green<|im_end|>

{'input_ids': (2, 22), 'attention_mask': (2, 22), 'labels': (2, 22)}


[{'position': 0,
  'id': 248045,
  'token': '<|im_start|>',
  'attention': 1,
  'label': -100,
  'supervised': False},
 {'position': 1,
  'id': 846,
  'token': 'user',
  'attention': 1,
  'label': -100,
  'supervised': False},
 {'position': 2,
  'id': 198,
  'token': 'Ċ',
  'attention': 1,
  'label': -100,
  'supervised': False},
 {'position': 3,
  'id': 3710,
  'token': 'What',
  'attention': 1,
  'label': -100,
  'supervised': False},
 {'position': 4,
  'id': 369,
  'token': 'Ġis',
  'attention': 1,
  'label': -100,
  'supervised': False},
 {'position': 5,
  'id': 279,
  'token': 'Ġthe',
  'attention': 1,
  'label': -100,
  'supervised': False},
 {'position': 6,
  'id': 1829,
  'token': 'Ġcolor',
  'attention': 1,
  'label': -100,
  'supervised': False},
 {'position': 7,
  'id': 314,
  'token': 'Ġof',
  'attention': 1,
  'label': -100,
  'supervised': False},
 {'position': 8,
  'id': 15879,
  'token': 'Ġgrass',
  'attention': 1,
  'label': -100,
  'supervised': False},
 {'position': 

## Other common local formats

Conversion rules belong close to the source data. Keep raw files immutable and write canonical outputs separately. Blank-line paragraphs are used as document boundaries here; for real data, use the original document IDs.


In [5]:
documents = [
    {"group_id": f"document-{i}", "text": paragraph.strip()}
    for i, paragraph in enumerate(
        (DATA_ROOT / "domain.txt").read_text(encoding="utf-8").split("\n\n")
    )
    if paragraph.strip()
]
instructions = [
    json.loads(line)
    for line in (DATA_ROOT / "instructions.jsonl").read_text(encoding="utf-8").splitlines()
]
converted = [
    {"prompt": row["instruction"] + "\n" + row["input"], "completion": row["output"]}
    for row in instructions
]
conversations = [
    json.loads(line)
    for line in (DATA_ROOT / "conversations.jsonl").read_text(encoding="utf-8").splitlines()
]
preferences = [
    {"prompt": row["prompt"], "chosen": row["chosen"], "rejected": row["rejected"]}
    for row in records
]
display(
    {
        "dapt": documents[0],
        "prompt_completion": converted[0],
        "conversation": conversations[0],
        "preference": preferences[0],
    }
)
display(token_table(tokenizer, collate_text(conversations)))

{'dapt': {'group_id': 'document-0', 'text': 'Water freezes at zero degrees .'},
 'prompt_completion': {'prompt': 'Answer briefly\nWhat is the color of sky ?',
  'completion': 'blue'},
 'conversation': {'group_id': 'dialogue-1',
  'messages': [{'role': 'user', 'content': 'What is the color of sky ?'},
   {'role': 'assistant', 'content': 'blue'},
   {'role': 'user', 'content': 'What is the color of grass ?'},
   {'role': 'assistant', 'content': 'green'}]},
 'preference': {'prompt': 'What is the color of sky ?',
  'chosen': 'blue',
  'rejected': 'green'}}

[{'position': 0,
  'id': 248045,
  'token': '<|im_start|>',
  'attention': 1,
  'label': -100,
  'supervised': False},
 {'position': 1,
  'id': 846,
  'token': 'user',
  'attention': 1,
  'label': -100,
  'supervised': False},
 {'position': 2,
  'id': 198,
  'token': 'Ċ',
  'attention': 1,
  'label': -100,
  'supervised': False},
 {'position': 3,
  'id': 3710,
  'token': 'What',
  'attention': 1,
  'label': -100,
  'supervised': False},
 {'position': 4,
  'id': 369,
  'token': 'Ġis',
  'attention': 1,
  'label': -100,
  'supervised': False},
 {'position': 5,
  'id': 279,
  'token': 'Ġthe',
  'attention': 1,
  'label': -100,
  'supervised': False},
 {'position': 6,
  'id': 1829,
  'token': 'Ġcolor',
  'attention': 1,
  'label': -100,
  'supervised': False},
 {'position': 7,
  'id': 314,
  'token': 'Ġof',
  'attention': 1,
  'label': -100,
  'supervised': False},
 {'position': 8,
  'id': 12515,
  'token': 'Ġsky',
  'attention': 1,
  'label': -100,
  'supervised': False},
 {'position': 9,

## Validate source isolation and load local split files

A filename does not guarantee a split is independent. Assert disjoint provenance and inspect duplicate content. The framework expects canonical records; it does not guess whether a column called `answer` means a completion or a verification target.


In [6]:
from finetunelab.config import RECIPE_ADAPTER
from finetunelab.data import load_datasets, validate_dataset

groups = {name: {r["group_id"] for r in rows} for name, rows in splits.items()}
assert not groups["train"] & groups["validation"]
assert not groups["train"] & groups["test"]
assert not groups["validation"] & groups["test"]
config = RECIPE_ADAPTER.validate_python(
    {
        "method": "sft",
        "model": {"name_or_path": str(MODEL_PATH), "local_files_only": True},
        "data": {
            "source": str(OUTPUT),
            "source_type": "local",
            "data_files": {name: f"{name}.jsonl" for name in splits},
            "eval_split": "validation",
        },
    }
)
train_dataset, validation_dataset = load_datasets(config)
print(validate_dataset(train_dataset, config))
assert len(validation_dataset) == len(splits["validation"])
assert (batch["labels"][batch["attention_mask"] == 0] == -100).all()
assert split_records(records + [records[0]]) == splits
previous_pad = tokenizer.pad_token
tokenizer.pad_token = tokenizer.eos_token
short = {"messages": [{"role": "user", "content": "one"}, {"role": "assistant", "content": "two"}]}
probe = collate_text([records[0], short])
assert (probe["labels"][probe["attention_mask"] == 0] == -100).all()
assert (probe["labels"] == tokenizer.eos_token_id).any(), "Real EOS must stay supervised."
tokenizer.pad_token = previous_pad

Generating train split: 10 examples [00:00, 2138.53 examples/s]
Generating validation split: 1 examples [00:00, 542.88 examples/s]
Generating test split: 1 examples [00:00, 793.62 examples/s]

{'sampled_rows': 10, 'method': 'sft'}


## Image annotations are references, not embeddings

Store a relative path with a canonical conversation. Resolve against a declared image root, decode to RGB, then let the checkpoint's processor choose patch geometry. Do not truncate the expanded visual token sequence. Lesson 07 performs the complete transformation.


In [7]:
from PIL import Image

image_root = OUTPUT / "images"
image_root.mkdir(exist_ok=True)
Image.new("RGB", (32, 32), "red").save(image_root / "red.png")
image_record = {
    "image": "red.png",
    "messages": [
        {
            "role": "user",
            "content": [{"type": "image"}, {"type": "text", "text": "What is the color ?"}],
        },
        {"role": "assistant", "content": "red"},
    ],
}
resolved = image_root / image_record["image"]
with Image.open(resolved) as image:
    print(image_record, image.convert("RGB").size)
(OUTPUT / "lesson_report.json").write_text(
    json.dumps({"splits": {k: len(v) for k, v in splits.items()}, "source_groups_disjoint": True}),
    encoding="utf-8",
)

{'image': 'red.png', 'messages': [{'role': 'user', 'content': [{'type': 'image'}, {'type': 'text', 'text': 'What is the color ?'}]}, {'role': 'assistant', 'content': 'red'}]} (32, 32)


85

## Common failures and exercises

- Missing required columns: inspect a raw record before mapping.
- Empty answer: reject it before training rather than producing an empty label mask.
- Dataset leakage: split by source group before chunking documents.
- Template mismatch: use the template distributed with the checkpoint and verify assistant boundaries.

**Exercises:** add an explicit CSV column mapping for your data; insert a duplicate conversation and show it is removed; inspect all supervised positions in a two-turn dialogue; change the image root and observe the missing-file diagnostic.
